# മൈക്രോസോഫ്റ്റ് ഏജന്റ് ഫ്രെയിംവർക്ക് — അസ്യൂർ ഓപ്പൺഎഐ (പ്രതികരണങ്ങൾ API)

ഈ കോഡ് സാമ്പിളിൽ, **മൈക്രോസോഫ്റ്റ് ഏജന്റ് ഫ്രെയിംവർക് (MAF)** ഉപയോഗിച്ച് **അസ്യൂർ ഓപ്പൺഎഐ** പിന്തുണയും ഉള്ള ഒരു ലഘു ഏജന്റ് **പ്രതികരണങ്ങൾ API** ഉപയോഗിച്ച് സൃഷ്ടിക്കും.

> **മൈഗ്രേഷൻ കുറിപ്പ്:** ഈ സാമ്പിൾ മുന്‍പ് ഗിറ്റ്ഹബ് മോഡലുകളോട് സെമാന്റിക്ക് കണല് ഉപയോഗിച്ചിരുന്നു. ഇത് മൈക്രോസോഫ്റ്റ് ഏജന്റ് ഫ്രെയിംവർക്ക് ലേക്ക് മാറ്റിയിട്ടുണ്ട്, ഗിറ്റ്ഹബ് മോഡലുകൾ (ഊർജ്ജം കുറഞ്ഞതും, ജൂലൈ 2026-ൽ വിരമിക്കുന്നതും) അസ്യൂർ ഓപ്പൺഎഐ ആയി മാറ്റപ്പെട്ടിട്ടുണ്ട്, ഇത് പ്രതികരണങ്ങൾ API-നെ പിന്തുണയ്ക്കുന്നു. MAF-യിലെ `OpenAIChatClient` അസ്യൂർ ഓപ്പൺഎഐയുടെ സ്ഥിരതയുള്ള `/openai/v1/` എന്റ്പോയിന്റിനെ ലക്ഷ്യമാക്കുന്നു, സാധാരണയായി പ്രതികരണങ്ങൾ API ഉപയോഗിക്കുന്നു.

ഇദ്ദേഹം ഈ സാമ്പിളിന്റെ ഉദ്ദേശ്യം പിന്നീട് വിവിധ ഏജൻറിക് പാറ്റേണുകൾ നടപ്പിലാക്കുമ്പോൾ എക്സ്ട്ര കോഡ് സാമ്പിളുകളിൽ പ്രയോഗിക്കാവുന്ന ഘട്ടങ്ങൾ വ്യക്തമാക്കുകയാണ്.


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## ആവശ്യമായ Python പാക്കേജുകൾ ഇറക്കുന്നിടം


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## ഉപകരണം നിർവചിക്കുന്നത്

Microsoft Agent Frameworkൽ, **ഉപകരണം** എന്നാൽ `@tool` കൊണ്ട് അലങ്കരിച്ച സാരളമായ Python ഫംഗ്ഷനാണ്, ഏജന്റ് വിളിക്കാവുന്നതാണ്. താഴെ ഒരു ര‍‍‍‍‍‍‍‍ണ്ടം അവധിക്കാല ടൂർഗതി തിരിച്ചറിയുന്ന ഉപകരണം നിർവചിച്ചിരിക്കുന്നു, അത് മുൻ പ്രാവശ്യം തിരഞ്ഞെടുത്തത് ആവർത്തിക്കുന്നത് ഒഴിവാക്കും.


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## ഏജന്റ് സൃഷ്ടിക്കല്‍

ഇവിടെ, നാം `TravelAgent` എന്ന പേരിലുള്ള ഏജന്റ് സൃഷ്ടിക്കുന്നു.

ഈ ഉദാഹരണത്തില്‍, നാം വളരെ അടിസ്ഥാനമായ നിര്‍ദ്ദേശങ്ങള്‍ ഉപയോഗിക്കുന്നു. ഏജന്റിന്റെ പെരുമാറ്റം എങ്ങനെ മാറുന്നുവെന്ന് കാണാന്‍ ഈ നിര്‍ദ്ദേശങ്ങള്‍ സ്വതന്ത്രമായി മാറ്റി പരിശോധിക്കുക.


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## ഏജന്റിനെ പ്രവർത്തിപ്പിക്കൽ

ഇപ്പോൾ നാം ഏജന്റ് പ്രവർത്തിപ്പിക്കാം. ഏജന്റിന് തിരുത്തലുകൾക്കിടയിൽ സംഭാഷണം ഓർമ്മമുണ്ടാക്കാൻ ഒരു `AgentSession` സൃഷ്ടിക്കുന്നു, പിന്നീട് രണ്ട് `user_inputs` അയക്കും. ആദ്യം ഒരു യാത്ര ചോദിക്കുന്നു; രണ്ടാമത്തത് ഉപഭോക്താവ് നിർദ്ദേശം ഇഷ്ടപ്പെടാത്തതായും മറ്റൊന്നിനായി ചോദിക്കുന്നതുമായതാണ് — ഏജന്റ് സെഷൻ ചരിത്രം കൂടാതെ `get_random_destination` ഉപകരണവും ഉപയോഗിച്ച് മറുപടി നൽകും.

ഏജന്റ് വ്യത്യസ്തമായി പ്രതികരിക്കുന്നത് കാണാൻ നിങ്ങൾ ഈ സന്ദേശങ്ങൾ മാറ്റിയാൽ മതി. മറുപടികൾ **ടോക്കൺ-ടോക്കണായി സ്രോതസ്** ആയി ലഭിക്കും.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
